In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 5.5
fig_height = 3.5
fig_format = 'pdf'
fig_dpi = 300
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L1VzZXJzL2RiaGFnaWEvTGlicmFyeS9DbG91ZFN0b3JhZ2UvRHJvcGJveC1DU1VGdWxsZXJ0b24vRGl2IEJoYWdpYS9UZWFjaGluZy9lY29uNTAyL2FkZC1jb250ZW50'
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/importlib/_bootstrap.py": 1764689462.0, "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/importlib/_bootstrap_external.py": 1764689462.0, "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/codecs.py": 1764689446.0, "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/encodings/aliases.py": 1764689466.0, "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/encodings/cp437.py": 1764689466.0, "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/encodings/__init__.py": 1764689466.0, "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/encodings/utf_8.py": 1764689467.0, "/Library/Developer/CommandLineT

In [2]:
#| label: fig-elasticity-two-panel
#| fig-cap: Two isoquants with the same MRTS at point A but different curvatures
#| echo: false

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# Wong colorblind-friendly palette
wong_blue = '#0072B2'
wong_orange = '#E69F00'
wong_green = '#009E73'
wong_vermillion = '#D55E00'
wong_purple = '#CC79A7'
wong_black = '#000000'

fig, axes = plt.subplots(1, 2, figsize=(7.5, 4), sharey=True)

L0, K0 = 3.0, 6.0
MRTS = 2.0
L1_high = 7.1
L1_low = 4.25

def ces_isoquant(L_arr, a, rho, q_target):
  K_vals = []
  for L in L_arr:
    inside = q_target**rho - a * L**rho
    if inside / (1 - a) > 0:
      K_vals.append((inside / (1 - a))**(1.0/rho))
    else:
      K_vals.append(np.nan)
  return np.array(K_vals)

# High sigma: sigma = 3 => rho = 2/3
rho_high = 1.0/2.0
a_high = 2**rho_high / (1 + 2**rho_high)
q_high = (a_high * L0**rho_high + (1 - a_high) * K0**rho_high)**(1.0/rho_high)
sigma_high = 1.0 / (1 - rho_high)

# Low sigma: sigma = 0.4 => rho = -1.5
rho_low = -1
a_low = 2**rho_low / (1 + 2**rho_low)
q_low = (a_low * L0**rho_low + (1 - a_low) * K0**rho_low)**(1.0/rho_low)
sigma_low = 1.0 / (1 - rho_low)

L_range = np.linspace(0.5, 10, 500)
K_high = ces_isoquant(L_range, a_high, rho_high, q_high)
K_low = ces_isoquant(L_range, a_low, rho_low, q_low)

# Move one unit: L = 3 -> L = 4
K1_h = ces_isoquant(np.array([L1_high]), a_high, rho_high, q_high)[0]
K1_l = ces_isoquant(np.array([L1_low]), a_low, rho_low, q_low)[0]

L_tan = np.linspace(1.0, 5.5, 100)
K_tan = K0 - MRTS * (L_tan - L0)

for idx, (K_iso, K1, L1, sigma_label, sigma_val) in enumerate([
  (K_high, K1_h, L1_high, 'High', sigma_high),
  (K_low, K1_l, L1_low, 'Low', sigma_low),
]):
  ax = axes[idx]

  # Isoquant
  mask = (K_iso > 0.2) & (K_iso < 14) & (~np.isnan(K_iso))
  ax.plot(L_range[mask], K_iso[mask], color=wong_blue, linewidth=2.5, label='Isoquant', zorder=2)

  # Points
  ax.plot(L0, K0, 'ko', markersize=9, zorder=6)
  ax.plot(L1, K1, 'D', color=wong_vermillion, markersize=9, zorder=6,
      markeredgecolor=wong_black, markeredgewidth=0.8)

  # Tangent lines at A and B
  L_tan_A = np.linspace(L0 - 1.5, L0 + 1.5, 100)
  K_tan_A = K0 - MRTS * (L_tan_A - L0)
  ax.plot(L_tan_A, K_tan_A, '--', color=wong_black, linewidth=1, alpha=1, zorder=1)


  # Labels
  ax.annotate('A', (L0 + 0.3, K0 + 0.5), fontsize=12, fontweight='bold')
  ax.annotate('B', (L1 + 0.2, K1 + 0.4), fontsize=12, fontweight='bold',
        color=wong_vermillion)

  # K/L ratio rays
  ax.plot([0, 8], [0, 8 * K0/L0], ':', color=wong_green, linewidth=1.2,
      alpha=1, label=f'K/L at A = {K0/L0:.1f}')
  ax.plot([0, 8], [0, 8 * K1/L1], ':', color=wong_orange, linewidth=1.2,
      alpha=1, label=f'K/L at B = {K1/L1:.1f}')

  ax.set_xlim(0, 9)
  ax.set_ylim(0, 13)
  ax.set_xlabel('Labor (L)', fontsize=12)
  ax.set_title(f'{sigma_label} elasticity of substitution\n(σ = {sigma_val})',
         fontsize=11)
  # show grid
  ax.grid(True, alpha=0.3)

# MRTS at all points
MRTS_A_high = a_high / (1 - a_high) * (K0/L0)**(1 - rho_high)
MRTS_B_high = a_high / (1 - a_high) * (K1_h/L1_high)**(1 - rho_high)
MRTS_A_low = a_low / (1 - a_low) * (K0/L0)**(1 - rho_low)
MRTS_B_low = a_low / (1 - a_low) * (K1_l/L1_low)**(1 - rho_low)


# Add tangent lines at A and B for both cases
L_tan_B_high = np.linspace(L1_high - 1.5, L1_high + 1.5, 100)
K_tan_B_high = K1_h - MRTS_B_high * (L_tan_B_high - L1_high)
L_tan_B_low = np.linspace(L1_low - 1.5, L1_low + 1.5, 100)
K_tan_B_low = K1_l - MRTS_B_low * (L_tan_B_low - L1_low)  
axes[0].plot(L_tan_B_high, K_tan_B_high, '--', color=wong_black, linewidth=1, alpha=1, zorder=1)
axes[1].plot(L_tan_B_low, K_tan_B_low, '--', color=wong_black, linewidth=1, alpha=1, zorder=1)


axes[0].set_ylabel('Capital (K)', fontsize=12)
plt.tight_layout()
plt.show()



# Table for both cases x point A and B, K/L and MRTS

data = {
  'Case': ['High σ', 'Low σ'],
  'σ': [sigma_high, sigma_low],
  'K/L at A': [K0/L0, K0/L0],
  'MRTS at A': [MRTS_A_high, MRTS_A_low],
  'K/L at B': [K1_h/L1_high, K1_l/L1_low],
  'MRTS at B': [MRTS_B_high, MRTS_B_low],
}
df = pd.DataFrame(data)
print(df.round(2).to_string(index=False))

<Figure size 2250x1200 with 2 Axes>

  Case   σ  K/L at A  MRTS at A  K/L at B  MRTS at B
High σ 2.0       2.0        2.0      0.18        0.6
 Low σ 0.5       2.0        2.0      1.09        0.6
